# Simple Collaborative Filtering with Matrix Factorization

This notebook trains a small user-item matrix factorization model on `ratings.csv` and uses it to recommend movies.

In [1]:
import numpy as np
import pandas as pd

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

print(f'ratings: {ratings.shape[0]:,} rows')
print(f'movies: {movies.shape[0]:,} rows')
ratings.head()

ratings: 100,836 rows
movies: 9,742 rows


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Prepare the rating matrix

We map the original `userId` and `movieId` values into dense row/column indexes. The model learns one vector for each user and one vector for each movie.

In [3]:
rng = np.random.default_rng(42)

user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
idx_to_movie = {idx: movie_id for movie_id, idx in movie_to_idx.items()}

ratings = ratings.assign(
    user_idx=ratings['userId'].map(user_to_idx),
    movie_idx=ratings['movieId'].map(movie_to_idx),
)

split_mask = rng.random(len(ratings)) < 0.8
train = ratings.loc[split_mask].reset_index(drop=True)
test = ratings.loc[~split_mask].reset_index(drop=True)

n_users = len(user_to_idx)
n_movies = len(movie_to_idx)
global_mean = train['rating'].mean()

print(f'users: {n_users:,}')
print(f'movies with ratings: {n_movies:,}')
print(f'train ratings: {len(train):,}')
print(f'test ratings: {len(test):,}')
print(f'global mean rating: {global_mean:.3f}')

users: 610
movies with ratings: 9,724
train ratings: 80,712
test ratings: 20,124
global mean rating: 3.502


In [4]:
print(train.head())
print(train.iloc[0])

   userId  movieId  rating  timestamp  user_idx  movie_idx
0       1        1     4.0  964982703         0          0
1       1        3     4.0  964981247         0          1
2       1       47     5.0  964983815         0          3
3       1       50     5.0  964982931         0          4
4       1      101     5.0  964980868         0          6
userId               1.0
movieId              1.0
rating               4.0
timestamp    964982703.0
user_idx             0.0
movie_idx            0.0
Name: 0, dtype: float64


## Train a simple matrix factorization model

Prediction formula:

`rating = global_mean + user_bias + movie_bias + dot(user_factors, movie_factors)`

The parameters are learned with stochastic gradient descent.

In [ ]:
rng = np.random.default_rng(seed=42)

# train_bias with bias
# returns user item embeddings and user and item bias
def train_bias(train, n_users, n_movies, k=3, epochs=10, lr=0.01, reg=0.05):
    # init user and item vectors
    user_factors = rng.normal(0, 0.1, size=(n_users, k)).astype(np.float32)
    item_factors = rng.normal(0, 0.1, size=(n_movies, k)).astype(np.float32)

    user_bias = np.zeros(n_users, dtype=np.float32)
    item_bias = np.zeros(n_movies, dtype=np.float32)

    global_mean = train["rating"].mean()

    for epoch in range(epochs):
        total_error = 0.0

        # for each rating, update both the user and item vectors
        # shuffle the training data at each epoch
        for row in train.sample(frac=1, random_state=epoch).itertuples():
            user_idx = row.user_idx
            movie_idx = row.movie_idx

            user_embed = user_factors[user_idx].copy()
            movie_embed = item_factors[movie_idx].copy()
            rating = row.rating

            # pred = global_mean + user_bias + item_bias + np.dot(user_embed, movie_embed)
            pred = global_mean + user_bias[user_idx] + item_bias[movie_idx] + np.dot(user_embed, movie_embed)
            # loss = 1/2 (rating - pred)^2
            error = rating - pred
            total_error += error ** 2

            # bu = bu + lr * (error - reg * bu)
            user_bias[user_idx] += lr * (error - reg * user_bias[user_idx])
            item_bias[movie_idx] += lr * (error - reg * item_bias[movie_idx])

            # ui = ui + lr * (error * vj - reg * ui)
            user_factors[user_idx] += lr * (error * movie_embed - reg * user_embed) # L2 regularization
            item_factors[movie_idx] += lr * (error * user_embed - reg * movie_embed)

        # report loss at each epoch
        print(f'epoch {epoch} RMSE : {np.sqrt(total_error / len(train)):.3f}')
    
    return user_factors, item_factors, user_bias, item_bias


## Evaluate the model

The baseline predicts the training-set average rating for every movie. The factor model should beat that simple baseline.

In [22]:
# evaluate with RMSE
def RMSE(preds, targets):
    return np.sqrt(np.mean((preds - targets) ** 2))

user_factors, item_factors, user_bias, item_bias = train_bias(train, n_users, n_movies, k=5, epochs=10, lr=0.01, reg=0.01)

# for each test rating, compute predicted rating and compute 
global_mean = train["rating"].mean()
test_preds = np.array([
    global_mean + user_bias[row.user_idx] + item_bias[row.movie_idx] + np.dot(user_factors[row.user_idx], item_factors[row.movie_idx])
    for row in test.itertuples()
])
test_rmse = RMSE(test_preds, test['rating'])

test_baseline = np.array([
    global_mean
    for _ in test.itertuples()
])
test_baseline_rmse = RMSE(test_baseline, test['rating'])

print(f'Baseline Test RMSE: {test_baseline_rmse:.3f}')
print(f'Test RMSE: {test_rmse:.3f}')


epoch 0 RMSE : 0.935
epoch 1 RMSE : 0.887
epoch 2 RMSE : 0.869
epoch 3 RMSE : 0.857
epoch 4 RMSE : 0.849
epoch 5 RMSE : 0.841
epoch 6 RMSE : 0.835
epoch 7 RMSE : 0.829
epoch 8 RMSE : 0.823
epoch 9 RMSE : 0.817
Baseline Test RMSE: 1.044
Test RMSE: 0.889


## Recommend movies for one user

For a selected user, score every unseen movie and return the top predicted ratings.

In [ ]:
movie_rating_counts = ratings.groupby('movieId').size()


def recommend_for_user(user_id, top_n=10, min_ratings=5):
    if user_id not in user_to_idx:
        raise ValueError(f'Unknown user_id: {user_id}')

    user_idx = user_to_idx[user_id]
    rated_movie_ids = set(ratings.loc[ratings['userId'] == user_id, 'movieId'])
    eligible_movie_ids = set(movie_rating_counts[movie_rating_counts >= min_ratings].index)

    candidate_movie_idx = np.array(
        [
            idx
            for idx, movie_id in idx_to_movie.items()
            if movie_id not in rated_movie_ids and movie_id in eligible_movie_ids
        ],
        dtype=np.int64,
    )
    candidate_user_idx = np.full(len(candidate_movie_idx), user_idx, dtype=np.int64)
    scores = predict(model, candidate_user_idx, candidate_movie_idx)

    top_positions = np.argsort(scores)[-top_n:][::-1]
    top_movie_ids = [idx_to_movie[idx] for idx in candidate_movie_idx[top_positions]]

    return (
        pd.DataFrame({'movieId': top_movie_ids, 'predicted_rating': scores[top_positions]})
        .merge(movies, on='movieId', how='left')
        [['title', 'genres', 'predicted_rating']]
    )


recommendations = recommend_for_user(user_id=1, top_n=10)
recommendations